### This script does basic bhv quantifications
#### this script runs DLC and quantifies the gazes
#### the following detailed analysis focused on pull related behavioral events
#### the pull action start events are defined based on the movement onset before each pull
#### capture the entire section of neural activity from Pull Start/Onset to pull actions

In [ ]:
import pandas as pd
import numpy as np
from numpy import genfromtxt
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.gridspec as gridspec
import seaborn
import scipy
import scipy.stats as st
import scipy.io
from sklearn.neighbors import KernelDensity
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples, silhouette_score
import string
import warnings
import pickle
import json

from scipy.ndimage import gaussian_filter1d

import os
import glob
import random
from time import time

from scipy.ndimage import label


### function - get body part location for each pair of cameras

In [ ]:
from ana_functions.body_part_locs_eachpair import body_part_locs_eachpair
from ana_functions.body_part_locs_singlecam import body_part_locs_singlecam

### function - align the two cameras

In [ ]:
from ana_functions.camera_align import camera_align       

### function - merge the two pairs of cameras

In [ ]:
from ana_functions.camera_merge import camera_merge

### function - find social gaze time point

In [ ]:
from ana_functions.find_socialgaze_timepoint import find_socialgaze_timepoint
from ana_functions.find_socialgaze_timepoint_singlecam import find_socialgaze_timepoint_singlecam
from ana_functions.find_socialgaze_timepoint_singlecam_wholebody import find_socialgaze_timepoint_singlecam_wholebody
from ana_functions.find_socialgaze_timepoint_singlecam_wholebody_2 import find_socialgaze_timepoint_singlecam_wholebody_2

### function - define time point of behavioral events

In [ ]:
from ana_functions.bhv_events_timepoint import bhv_events_timepoint
from ana_functions.bhv_events_timepoint_singlecam import bhv_events_timepoint_singlecam

### function - plot behavioral events

In [ ]:
from ana_functions.plot_bhv_events import plot_bhv_events
from ana_functions.plot_bhv_events_levertube import plot_bhv_events_levertube
from ana_functions.plot_continuous_bhv_var_singlecam_PullStartToPull_variedSection import plot_continuous_bhv_var_singlecam_PullStartToPull_variedSection
from ana_functions.plot_continuous_bhv_var_singlecam_PullStartToPull_variedSection_highbhvDimension_to_lowPCspace import plot_continuous_bhv_var_singlecam_PullStartToPull_variedSection_highbhvDimension_to_lowPCspace

from ana_functions.draw_self_loop import draw_self_loop
import matplotlib.patches as mpatches 
from matplotlib.collections import PatchCollection

### function - plot inter-pull interval

In [ ]:
from ana_functions.plot_interpull_interval import plot_interpull_interval

### function - make demo videos with skeleton and inportant vectors

In [ ]:
from ana_functions.tracking_video_singlecam_demo import tracking_video_singlecam_demo
from ana_functions.tracking_video_singlecam_wholebody_demo import tracking_video_singlecam_wholebody_demo

### function - interval between all behavioral events

In [ ]:
from ana_functions.bhv_events_interval import bhv_events_interval

### function - other useful functions

In [ ]:
# get useful information about pulls
from ana_functions.get_pull_infos import get_pull_infos

In [ ]:
# for projecting high D bhv variables to small PC space
from ana_functions.singlecam_conBhv_from_highDimension_to_PCspace import get_data_for_singlecam_conBhv_from_highDimension_to_PCspace

In [ ]:
# use the gaze vector speed and face mass speed to find the pull action start time within IPI
from ana_functions.find_sharp_increases_withinIPI import find_sharp_increases_withinIPI
from ana_functions.find_sharp_increases_withinIPI import find_sharp_increases_withinIPI_dual_speed

In [ ]:
# method 2: find the lowest timepoint then the increase point as the pull onset
from ana_functions.find_rising_onset_after_min_withinIPI import find_rising_onset_after_min_withinIPI
from ana_functions.find_rising_onset_after_min_withinIPI import find_rising_onset_after_min_dual_speed

## Analyze each session

### prepare the basic behavioral data (especially the time stamps for each bhv events)
### separate each session based on trial types (different force levels)

In [ ]:
# instead of using gaze angle threshold, use the target rectagon to deside gaze info
# ...need to update
sqr_thres_tubelever = 75 # draw the square around tube and lever
sqr_thres_face = 1.15 # a ratio for defining face boundary
sqr_thres_body = 4 # how many times to enlongate the face box boundry to the body


# get the fps of the analyzed video
fps = 30

# frame number of the demo video
# nframes = 0.5*30 # second*30fps
nframes = 1*30 # second*30fps

# re-analyze the video or not
reanalyze_video = 0
redo_anystep = 0

# only analyze the best (five) sessions for each conditions
do_bestsession = 1
if do_bestsession:
    savefile_sufix = '_bestsessions'
else:
    savefile_sufix = '_allsessions'
    
    
# if use onset of the first increase after min
doOnsetAfterMin = 1
if not doOnsetAfterMin:
    doOnsetAfterMin_suffix = 'bhvPCA_'
elif doOnsetAfterMin:
    doOnsetAfterMin_suffix = 'bhvPCA_PullOnsetAfterMin_'

# if use a hmm based method to find the trial start
doHMMmethod = 0
if doHMMmethod:
    doOnsetAfterMin_suffix = 'HMMmethods_'
    
    
# all the videos (no misaligned ones)
# aligned with the audio
# get the session start time from "videosound_bhv_sync.py/.ipynb"
# currently the session_start_time will be manually typed in. It can be updated after a better method is used

# force manipulation type
# SR_bothchange: self reward, both forces changed
# CO_bothchange: 1s cooperation, both forces changed
# CO_A1change: 1s cooperation, animal 1 forces changed
# CO_A2change: 1s cooperation, animal 2 forces changed
forceManiType = 'CO_A1change'

# Koala Vermelho
if 1:
    if do_bestsession:      
        # both animals' lever force were changed - Self reward
        if forceManiType == 'SR_bothchange':
            dates_list = [ "20240228","20240229","20240409","20240411",
                           "20240412","20240416","20240419",] 
            session_start_times = [ 64.5,  73.5,  0.00,  0.00,  
                                    0.00,  0.00,  0.00,  ] # in second
        # both animals' lever force were changed - cooperation
        elif forceManiType == 'CO_bothchange':
            dates_list = [ "20240304", ]
            session_start_times = [ 0.00, ] # in second
        # Koala's lever force were changed
        if forceManiType == 'CO_A1change':
            dates_list = [ "20240305","20240306","20240313","20240318","20240321",
                           "20240426","20240429","20240430",]
            session_start_times = [ 62.0,  55.2,  0.00,  0.00,  0.00, 
                                    0.00,  0.00,  0.00,  ] # in second
        # Verm's lever force were changed
        if forceManiType == 'CO_A2change':
            dates_list = [ "20240307","20240308","20240311","20240319",
                           "20240320","20240422","20240423","20240425",
                           "20240621", ]
            session_start_times = [ 72.2,  0.00,  60.8,  0.00,  
                                    0.00,  53.0,  0.00,  0.00, 
                                    0.00, ] # in second       
    
    elif not do_bestsession:
        # pick only five sessions for each conditions
        dates_list = [
                    
                     ]
        session_start_times = [ 
                               
                              ] # in second
    
    animal1_fixedorder = ['koala']
    animal2_fixedorder = ['vermelho']

    animal1_filename = "Koala"
    animal2_filename = "Vermelho"
    

# Dannon Kanga
if 0:
    if do_bestsession:      
        # both animals' lever force were changed - Self reward
        if forceManiType == 'SR_bothchange':
            dates_list = [ "20240912","20240913","20240917","20241101","20241104",
                           "20241105",
                           ] 
            session_start_times = [ 0.00,  0.00, 0.00, 0.00, 0.00,
                                    0.00,
                                   ] # in second
        # both animals' lever force were changed - cooperation
        elif forceManiType == 'CO_bothchange':
            dates_list = [  ]
            session_start_times = [  ] # in second
        # Dannon's lever force were changed
        if forceManiType == 'CO_A1change':
            dates_list = [ "20241009","20241011","20241016","20241018","20241022", 
                           "20241025", ]
            session_start_times = [ 0.00, 0.00, 0.00, 0.00, 0.00, 
                                    0.00, ] # in second
        # Kanga's lever force were changed
        if forceManiType == 'CO_A2change':
            dates_list = [ "20240910","20240911","20240916","20240918","20240919" ,
                           "20241008","20241010","20241014","20241017"]
            session_start_times = [ 0.00, 0.00, 0.00, 43.5, 0.00, 
                                    59.6, 66.0, 0.00, 0.00, ] # in second       
    
    elif not do_bestsession:
        # pick only five sessions for each conditions
        dates_list = [
                      
                     ]
        session_start_times = [ 
                               
                              ] # in second
    
    animal1_fixedorder = ['dannon']
    animal2_fixedorder = ['kanga']

    animal1_filename = "Dannon"
    animal2_filename = "Kanga"
    
    
# Dodon Kanga
if 0:
    if do_bestsession:      
        # both animals' lever force were changed - Self reward
        if forceManiType == 'SR_bothchange':
            dates_list = [ "20250505", "20250506_SR", "20250508_SR", "20250516_SR",
                           "20250519", "20250520_SR", "20250522_SR", "20250523_SR",
                         ] 
            session_start_times = [ 147.1, 0.00, 0.00, 0.00,
                                    106.0, 0.00, 0.00, 257.0,    
                                  ] # in second
        # both animals' lever force were changed - cooperation
        elif forceManiType == 'CO_bothchange':
            dates_list = [  ]
            session_start_times = [  ] # in second
        # Dodson's lever force were changed
        if forceManiType == 'CO_A1change':
            dates_list = [ "20250506_MC", "20250508", "20250512_MC", "20250520",
                           "20250523", 
                         ]
            session_start_times = [ 0.00, 151.0, 265.0, 0.00, 
                                    66.4,
                                  ] # in second
        # Kanga's lever force were changed
        if forceManiType == 'CO_A2change':
            dates_list = [ "20250507", "20250516_MC", "20250522" ]
            session_start_times = [ 0.00, 171.1, 0.00,  ] # in second       
    
    elif not do_bestsession:
        # pick only five sessions for each conditions
        dates_list = [
                      
                     ]
        session_start_times = [ 
                               
                              ] # in second
    
    animal1_fixedorder = ['dodson']
    animal2_fixedorder = ['kanga']

    animal1_filename = "Dodson"
    animal2_filename = "Kanga"
    
#    
# dates_list = ["20240430"]
# session_start_times = [0.00] # in second
ndates = np.shape(dates_list)[0]

session_start_frames = session_start_times * fps # fps is 30Hz

totalsess_time = 600

# video tracking results info
animalnames_videotrack = ['dodson','scorch'] # does not really mean dodson and scorch, instead, indicate animal1 and animal2
bodypartnames_videotrack = ['rightTuft','whiteBlaze','leftTuft','rightEye','leftEye','mouth']


# which camera to analyzed
cameraID = 'camera-2'
cameraID_short = 'cam2'


# location of levers and tubes for camera 2
# get this information using DLC animal tracking GUI, the results are stored: 
# /home/ws523/marmoset_tracking_DLCv2/marmoset_tracking_with_lever_tube-weikang-2023-04-13/labeled-data/
considerlevertube = 1
considertubeonly = 0
# # camera 1
# lever_locs_camI = {'dodson':np.array([645,600]),'scorch':np.array([425,435])}
# tube_locs_camI  = {'dodson':np.array([1350,630]),'scorch':np.array([555,345])}
# # camera 2
lever_locs_camI = {'dodson':np.array([1335,715]),'scorch':np.array([550,715])}
tube_locs_camI  = {'dodson':np.array([1550,515]),'scorch':np.array([350,515])}
# # lever_locs_camI = {'dodson':np.array([1335,715]),'scorch':np.array([550,715])}
# # tube_locs_camI  = {'dodson':np.array([1650,490]),'scorch':np.array([250,490])}
# # camera 3
# lever_locs_camI = {'dodson':np.array([1580,440]),'scorch':np.array([1296,540])}
# tube_locs_camI  = {'dodson':np.array([1470,375]),'scorch':np.array([805,475])}


if np.shape(session_start_times)[0] != np.shape(dates_list)[0]:
    exit()

    
# define bhv events summarizing variables  
# align the animal1 and animal2 across the sessions
animal1_name_all_dates = np.empty(shape=(0,), dtype=str)
animal2_name_all_dates = np.empty(shape=(0,), dtype=str)
trialdates_all_dates = np.empty(shape=(0,), dtype=str)
tasktypes_all_dates = np.zeros((0,))
coopthres_all_dates = np.zeros((0,))
force1_all_dates = np.zeros((0,)) 
force2_all_dates = np.zeros((0,)) 

subblockID_all_dates = np.zeros((0,))

succ_rate_all_dates = np.zeros((0,))
succ_rate1_all_dates = np.zeros((0,))
succ_rate2_all_dates = np.zeros((0,))

trialnum_all_dates = np.zeros((0,))
blocktime_all_dates = np.zeros((0,))

interpullintv_all_dates = np.zeros((0,))
pull1_IPI_all_dates = np.zeros((0,))
pull2_IPI_all_dates = np.zeros((0,))
pull1_IPI_std_all_dates = np.zeros((0,))
pull2_IPI_std_all_dates = np.zeros((0,))

owgaze1_num_all_dates = np.zeros((0,))
owgaze2_num_all_dates = np.zeros((0,))
mtgaze1_num_all_dates = np.zeros((0,))
mtgaze2_num_all_dates = np.zeros((0,))
pull1_num_all_dates = np.zeros((0,))
pull2_num_all_dates = np.zeros((0,))

#
pull_rts_all_dates = dict.fromkeys(dates_list, [])

pull_infos_all_dates = dict.fromkeys(dates_list, []) # keep some useful information about pulls - time from last reward, number of preceding failed pull, force conditions


bhvtoPC1_loadings_all_dates = dict.fromkeys(dates_list, [])
bhvPC123explained_all_dates = dict.fromkeys(dates_list, [])

pullstartTopull_trig_events_all_dates = dict.fromkeys(dates_list, [])
succpullstartTopull_trig_events_all_dates = dict.fromkeys(dates_list, [])
failpullstartTopull_trig_events_all_dates = dict.fromkeys(dates_list, [])

# bhvevents_pullstartTopull_aligned_FR_allevents_all_dates = dict.fromkeys(dates_list, [])

# where to save the summarizing data
data_saved_folder = '/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/3d_recontruction_analysis_forceManipulation_task_data_saved/'


    

In [ ]:
# basic behavior analysis (define time stamps for each bhv events, etc)

try:
    if redo_anystep:
        dummy
    
    # dummy
    
    # load saved data
    data_saved_subfolder = data_saved_folder+'data_saved_singlecam_wholebody'+savefile_sufix+'/'+cameraID+'/'+animal1_fixedorder[0]+animal2_fixedorder[0]+'/'
   
    with open(data_saved_subfolder+'/animal1_name_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'rb') as f:
        animal1_name_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/animal2_name_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'rb') as f:
        animal2_name_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/trialdates_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'rb') as f:
        trialdates_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/tasktypes_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'rb') as f:
        tasktypes_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/coopthres_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'rb') as f:
        coopthres_all_dates = pickle.load(f)

    with open(data_saved_subfolder+'/force1_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'rb') as f:
        force1_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/force2_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'rb') as f:
        force2_all_dates = pickle.load(f)

    with open(data_saved_subfolder+'/subblockID_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'rb') as f:
        subblockID_all_dates = pickle.load(f)

    with open(data_saved_subfolder+'/succ_rate_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'rb') as f:
        succ_rate_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/succ_rate1_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'rb') as f:
        succ_rate1_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/succ_rate2_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'rb') as f:
        succ_rate2_all_dates = pickle.load(f)
        
    with open(data_saved_subfolder+'/trialnum_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'rb') as f:
        trialnum_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/blocktime_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'rb') as f:
        blocktime_all_dates = pickle.load(f)

    with open(data_saved_subfolder+'/interpullintv_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'rb') as f:
        interpullintv_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/pull1_IPI_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'rb') as f:
        pull1_IPI_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/pull1_IPI_std_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'rb') as f:
        pull1_IPI_std_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/pull2_IPI_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'rb') as f:
        pull2_IPI_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/pull2_IPI_std_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'rb') as f:
        pull2_IPI_std_all_dates = pickle.load(f)

    with open(data_saved_subfolder+'/owgaze1_num_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'rb') as f:
        owgaze1_num_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/owgaze2_num_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'rb') as f:
        owgaze2_num_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/mtgaze1_num_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'rb') as f:
        mtgaze1_num_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/mtgaze2_num_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'rb') as f:
        mtgaze2_num_all_dates = pickle.load(f)     
    with open(data_saved_subfolder+'/pull1_num_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'rb') as f:
        pull1_num_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/pull2_num_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'rb') as f:
        pull2_num_all_dates = pickle.load(f)
        
    # 
    with open(data_saved_subfolder+'/pullstartTopull_trig_events_all_dates_'+doOnsetAfterMin_suffix+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'rb') as f:
        pullstartTopull_trig_events_all_dates = pickle.load(f) 
    with open(data_saved_subfolder+'/succpullstartTopull_trig_events_all_dates_'+doOnsetAfterMin_suffix+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'rb') as f:
        succpullstartTopull_trig_events_all_dates = pickle.load(f) 
    with open(data_saved_subfolder+'/failpullstartTopull_trig_events_all_dates_'+doOnsetAfterMin_suffix+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'rb') as f:
        failpullstartTopull_trig_events_all_dates = pickle.load(f) 
    
        
    with open(data_saved_subfolder+'/bhvtoPC1_loadings_all_dates_'+doOnsetAfterMin_suffix+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'rb') as f:
        bhvtoPC1_loadings_all_dates  = pickle.load(f)
        
    with open(data_saved_subfolder+'/bhvPC123explained_all_dates_'+doOnsetAfterMin_suffix+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'rb') as f:
        bhvPC123explained_all_dates  = pickle.load(f)
        
    with open(data_saved_subfolder+'/pull_rts_all_dates_'+doOnsetAfterMin_suffix+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'rb') as f:
        pull_rts_all_dates  = pickle.load(f)
    
    with open(data_saved_subfolder+'/pull_infos_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'rb') as f:
        pull_infos_all_dates  = pickle.load(f)  
        
    
    print('all data from all dates are loaded')

except:

    print('analyze all dates')

    for idate in np.arange(0,ndates,1):
        date_tgt = dates_list[idate]
        session_start_time = session_start_times[idate]
        
        #
        pull_rts_all_dates[date_tgt] = dict.fromkeys([animal1_fixedorder[0],animal2_fixedorder[0]],[])
        
        #
        bhvtoPC1_loadings_all_dates[date_tgt] = dict.fromkeys([animal1_fixedorder[0],animal2_fixedorder[0]],[])
        bhvPC123explained_all_dates[date_tgt] = dict.fromkeys([animal1_fixedorder[0],animal2_fixedorder[0]],[])
        
        # load behavioral results
        try:
            bhv_data_path = "/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/marmoset_tracking_bhv_data_forceManipulation_task/"+date_tgt+"_"+animal1_filename+"_"+animal2_filename+"/"
            trial_record_json = glob.glob(bhv_data_path +date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_TrialRecord_" + "*.json")
            bhv_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_bhv_data_" + "*.json")
            session_info_json = glob.glob(bhv_data_path + date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_session_info_" + "*.json")
            lever_reading_json = glob.glob(bhv_data_path + date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_lever_reading_" + "*.json") 
            #
            trial_record = pd.read_json(trial_record_json[0])
            bhv_data = pd.read_json(bhv_data_json[0])
            session_info = pd.read_json(session_info_json[0])
            lever_reading = pd.read_json(lever_reading_json[0])
        except:
            bhv_data_path = "/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/marmoset_tracking_bhv_data_forceManipulation_task/"+date_tgt+"_"+animal1_filename+"_"+animal2_filename+"/"
            trial_record_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_TrialRecord_" + "*.json")
            bhv_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_bhv_data_" + "*.json")
            session_info_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_session_info_" + "*.json")
            lever_reading_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_lever_reading_" + "*.json")             
            #
            trial_record = pd.read_json(trial_record_json[0])
            bhv_data = pd.read_json(bhv_data_json[0])
            session_info = pd.read_json(session_info_json[0])
            lever_reading = pd.read_json(lever_reading_json[0])

        # get animal info from the session information
        animal1 = session_info['lever1_animal'][0].lower()
        animal2 = session_info['lever2_animal'][0].lower()
        
        # successful trial or not
        succtrial_ornot = np.array((trial_record['rewarded']>0).astype(int))
        succpull1_ornot = np.array((np.isin(bhv_data[bhv_data['behavior_events']==1]['trial_number'],trial_record[trial_record['rewarded']>0]['trial_number'])).astype(int))
        succpull2_ornot = np.array((np.isin(bhv_data[bhv_data['behavior_events']==2]['trial_number'],trial_record[trial_record['rewarded']>0]['trial_number'])).astype(int))
        succpulls_ornot = [succpull1_ornot,succpull2_ornot]

        # clean up the trial_record
        warnings.filterwarnings('ignore')
        trial_record_clean = pd.DataFrame(columns=trial_record.columns)
        for itrial in np.arange(0,np.max(trial_record['trial_number']),1):
            # trial_record_clean.loc[itrial] = trial_record[trial_record['trial_number']==itrial+1].iloc[[0]]
            trial_record_clean = trial_record_clean.append(trial_record[trial_record['trial_number']==itrial+1].iloc[[0]])
        trial_record_clean = trial_record_clean.reset_index(drop = True)

        # change bhv_data time to the absolute time
        time_points_new = pd.DataFrame(np.zeros(np.shape(bhv_data)[0]),columns=["time_points_new"])
        for itrial in np.arange(0,np.max(trial_record_clean['trial_number']),1):
            ind = bhv_data["trial_number"]==itrial+1
            new_time_itrial = bhv_data[ind]["time_points"] + trial_record_clean["trial_starttime"].iloc[itrial]
            time_points_new["time_points_new"][ind] = new_time_itrial
        bhv_data["time_points"] = time_points_new["time_points_new"]
        bhv_data = bhv_data[bhv_data["time_points"] != 0]

        # change lever reading time to the absolute time
        time_points_new = pd.DataFrame(np.zeros(np.shape(lever_reading)[0]),columns=["time_points_new"])
        for itrial in np.arange(0,np.max(trial_record_clean['trial_number']),1):
            ind = lever_reading["trial_number"]==itrial+1
            new_time_itrial = lever_reading[ind]["readout_timepoint"] + trial_record_clean["trial_starttime"].iloc[itrial]
            time_points_new["time_points_new"][ind] = new_time_itrial
        lever_reading["readout_timepoint"] = time_points_new["time_points_new"]
        lever_reading = lever_reading[lever_reading["readout_timepoint"] != 0]
        #
        lever1_pull = lever_reading[(lever_reading['lever_id']==1)&(lever_reading['pull_or_release']==1)]
        lever1_release = lever_reading[(lever_reading['lever_id']==1)&(lever_reading['pull_or_release']==0)]
        lever2_pull = lever_reading[(lever_reading['lever_id']==2)&(lever_reading['pull_or_release']==1)]
        lever2_release = lever_reading[(lever_reading['lever_id']==2)&(lever_reading['pull_or_release']==0)]
        #
        if np.shape(lever1_release)[0]<np.shape(lever1_pull)[0]:
            lever1_pull = lever1_pull.iloc[0:-1]
        if np.shape(lever2_release)[0]<np.shape(lever2_pull)[0]:
            lever2_pull = lever2_pull.iloc[0:-1]
        #
        lever1_pull_release = lever1_pull
        lever1_pull_release['delta_timepoint'] = np.array(lever1_release['readout_timepoint'].reset_index(drop=True)-lever1_pull['readout_timepoint'].reset_index(drop=True))
        lever1_pull_release['delta_gauge'] = np.array(lever1_release['strain_gauge'].reset_index(drop=True)-lever1_pull['strain_gauge'].reset_index(drop=True))
        lever2_pull_release = lever2_pull
        lever2_pull_release['delta_timepoint'] = np.array(lever2_release['readout_timepoint'].reset_index(drop=True)-lever2_pull['readout_timepoint'].reset_index(drop=True))
        lever2_pull_release['delta_gauge'] = np.array(lever2_release['strain_gauge'].reset_index(drop=True)-lever2_pull['strain_gauge'].reset_index(drop=True))
        
        
        # load behavioral event results from the tracking analysis
        if 1:
            # folder and file path
            camera12_analyzed_path = "/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/test_video_forceManipulation_task_3d/"+date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_camera12/"
            camera23_analyzed_path = "/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/test_video_forceManipulation_task_3d/"+date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_camera23/"

            try:
                singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_cameraSep1shuffle1_150000"
                try: 
                    bodyparts_camI_camIJ = camera12_analyzed_path+date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
                    # get the bodypart data from files
                    bodyparts_locs_camI = body_part_locs_singlecam(bodyparts_camI_camIJ,singlecam_ana_type,animalnames_videotrack,bodypartnames_videotrack,date_tgt)
                    video_file_original = camera12_analyzed_path+date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+".mp4"
                except:
                    bodyparts_camI_camIJ = camera23_analyzed_path+date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
                    # get the bodypart data from files
                    bodyparts_locs_camI = body_part_locs_singlecam(bodyparts_camI_camIJ,singlecam_ana_type,animalnames_videotrack,bodypartnames_videotrack,date_tgt)
                    video_file_original = camera23_analyzed_path+date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+".mp4"        
            except:
                singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_camera_withHeadchamberFeb28shuffle1_167500"
                try: 
                    bodyparts_camI_camIJ = camera12_analyzed_path+date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
                    # get the bodypart data from files
                    bodyparts_locs_camI = body_part_locs_singlecam(bodyparts_camI_camIJ,singlecam_ana_type,animalnames_videotrack,bodypartnames_videotrack,date_tgt)
                    video_file_original = camera12_analyzed_path+date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+".mp4"
                except:
                    bodyparts_camI_camIJ = camera23_analyzed_path+date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
                    # get the bodypart data from files
                    bodyparts_locs_camI = body_part_locs_singlecam(bodyparts_camI_camIJ,singlecam_ana_type,animalnames_videotrack,bodypartnames_videotrack,date_tgt)
                    video_file_original = camera23_analyzed_path+date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+".mp4"        
                              
                    
            try:
                if reanalyze_video:
                    dummy
                print('load social gaze with '+cameraID+' only of '+date_tgt)
                with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_look_ornot.pkl', 'rb') as f:
                    output_look_ornot = pickle.load(f)
                with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_allvectors.pkl', 'rb') as f:
                    output_allvectors = pickle.load(f)
                with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_allangles.pkl', 'rb') as f:
                    output_allangles = pickle.load(f)  
                with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_key_locations.pkl', 'rb') as f:
                    output_key_locations = pickle.load(f)
                
            except:   
                print('analyze social gaze with '+cameraID+' only of '+date_tgt)
                # get social gaze information 
                output_look_ornot, output_allvectors, output_allangles = find_socialgaze_timepoint_singlecam_wholebody(bodyparts_locs_camI,lever_locs_camI,tube_locs_camI,
                                                                                                                       considerlevertube,considertubeonly,sqr_thres_tubelever,
                                                                                                                      sqr_thres_face,sqr_thres_body)
                
                output_key_locations = find_socialgaze_timepoint_singlecam_wholebody_2(bodyparts_locs_camI,lever_locs_camI,tube_locs_camI,considerlevertube)

                # save data
                current_dir = data_saved_folder+'/bhv_events_singlecam_wholebody/'+animal1_fixedorder[0]+animal2_fixedorder[0]
                add_date_dir = os.path.join(current_dir,cameraID+'/'+date_tgt)
                if not os.path.exists(add_date_dir):
                    os.makedirs(add_date_dir)
                #
                with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_look_ornot.pkl', 'wb') as f:
                    pickle.dump(output_look_ornot, f)
                with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_allvectors.pkl', 'wb') as f:
                    pickle.dump(output_allvectors, f)
                with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_allangles.pkl', 'wb') as f:
                    pickle.dump(output_allangles, f)
                with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_key_locations.pkl', 'wb') as f:
                    pickle.dump(output_key_locations, f)


            look_at_other_or_not_merge = output_look_ornot['look_at_other_or_not_merge']
            look_at_tube_or_not_merge = output_look_ornot['look_at_tube_or_not_merge']
            look_at_lever_or_not_merge = output_look_ornot['look_at_lever_or_not_merge']
            # change the unit to second
            session_start_time = session_start_times[idate]
            look_at_other_or_not_merge['time_in_second'] = np.arange(0,np.shape(look_at_other_or_not_merge['dodson'])[0],1)/fps - session_start_time
            look_at_lever_or_not_merge['time_in_second'] = np.arange(0,np.shape(look_at_lever_or_not_merge['dodson'])[0],1)/fps - session_start_time
            look_at_tube_or_not_merge['time_in_second'] = np.arange(0,np.shape(look_at_tube_or_not_merge['dodson'])[0],1)/fps - session_start_time 

            # find time point of behavioral events
            output_time_points_socialgaze ,output_time_points_levertube = bhv_events_timepoint_singlecam(bhv_data,look_at_other_or_not_merge,look_at_lever_or_not_merge,look_at_tube_or_not_merge)
            time_point_pull1 = output_time_points_socialgaze['time_point_pull1']
            time_point_pull2 = output_time_points_socialgaze['time_point_pull2']
            oneway_gaze1 = output_time_points_socialgaze['oneway_gaze1']
            oneway_gaze2 = output_time_points_socialgaze['oneway_gaze2']
            mutual_gaze1 = output_time_points_socialgaze['mutual_gaze1']
            mutual_gaze2 = output_time_points_socialgaze['mutual_gaze2']
            
            # get the force level of each pull
            bhv_data_moreinfo = bhv_data.merge(
                            trial_record_clean[['trial_number', 'lever1_force', 'lever2_force']],
                            on='trial_number',
                            how='left')
            force_pull1 = bhv_data_moreinfo["lever1_force"][bhv_data_moreinfo["behavior_events"]==1]
            partnerforce_pull1 = bhv_data_moreinfo["lever2_force"][bhv_data_moreinfo["behavior_events"]==1]
            force_pull2 = bhv_data_moreinfo["lever2_force"][bhv_data_moreinfo["behavior_events"]==2]
            partnerforce_pull2 = bhv_data_moreinfo["lever1_force"][bhv_data_moreinfo["behavior_events"]==2]
              
            
            # new total session time (instead of 600s) - total time of the video recording
            totalsess_time = np.floor(np.shape(output_look_ornot['look_at_lever_or_not_merge']['dodson'])[0]/30) 

            # make sure all task code registered event, aka pulls, are within the video recording
            ind_good_pull1 = time_point_pull1 < (totalsess_time - session_start_time)
            time_point_pull1 = time_point_pull1[ind_good_pull1]
            force_pull1 = force_pull1[ind_good_pull1]
            partnerforce_pull1 = partnerforce_pull1[ind_good_pull1]
            #
            ind_good_pull2 = time_point_pull2 < (totalsess_time - session_start_time)
            time_point_pull2 = time_point_pull2[ind_good_pull2]
            force_pull2 = force_pull2[ind_good_pull2]
            partnerforce_pull2 = partnerforce_pull2[ind_good_pull2]
            
            #
            # define successful pulls and failed pulls
            if 0: # old definition; not in use
                trialnum_succ = np.array(trial_record_clean['trial_number'][trial_record_clean['rewarded']>0])
                bhv_data_succ = bhv_data[np.isin(bhv_data['trial_number'],trialnum_succ)]
                #
                time_point_pull1_succ = bhv_data_succ["time_points"][bhv_data_succ["behavior_events"]==1]
                time_point_pull2_succ = bhv_data_succ["time_points"][bhv_data_succ["behavior_events"]==2]
                time_point_pull1_succ = np.round(time_point_pull1_succ,1)
                time_point_pull2_succ = np.round(time_point_pull2_succ,1)
                #
                trialnum_fail = np.array(trial_record_clean['trial_number'][trial_record_clean['rewarded']==0])
                bhv_data_fail = bhv_data[np.isin(bhv_data['trial_number'],trialnum_fail)]
                #
                time_point_pull1_fail = bhv_data_fail["time_points"][bhv_data_fail["behavior_events"]==1]
                time_point_pull2_fail = bhv_data_fail["time_points"][bhv_data_fail["behavior_events"]==2]
                time_point_pull1_fail = np.round(time_point_pull1_fail,1)
                time_point_pull2_fail = np.round(time_point_pull2_fail,1)
            else:
                # a new definition of successful and failed pulls
                # separate successful and failed pulls
                # step 1 all pull and juice
                time_point_pull1 = bhv_data["time_points"][bhv_data["behavior_events"]==1]
                time_point_pull2 = bhv_data["time_points"][bhv_data["behavior_events"]==2]
                time_point_juice1 = bhv_data["time_points"][bhv_data["behavior_events"]==3]
                time_point_juice2 = bhv_data["time_points"][bhv_data["behavior_events"]==4]
                # step 2:
                # pull 1
                # Find the last pull before each juice
                successful_pull1 = [time_point_pull1[time_point_pull1 < juice].max() for juice in time_point_juice1]
                # Convert to Pandas Series
                successful_pull1 = pd.Series(successful_pull1, index=time_point_juice1.index)
                # Find failed pulls (pulls that are not successful)
                failed_pull1 = time_point_pull1[~time_point_pull1.isin(successful_pull1)]
                # pull 2
                # Find the last pull before each juice
                successful_pull2 = [time_point_pull2[time_point_pull2 < juice].max() for juice in time_point_juice2]
                # Convert to Pandas Series
                successful_pull2 = pd.Series(successful_pull2, index=time_point_juice2.index)
                # Find failed pulls (pulls that are not successful)
                failed_pull2 = time_point_pull2[~time_point_pull2.isin(successful_pull2)]
                #
                # step 3:
                time_point_pull1_succ = np.round(successful_pull1,1)
                time_point_pull2_succ = np.round(successful_pull2,1)
                time_point_pull1_fail = np.round(failed_pull1,1)
                time_point_pull2_fail = np.round(failed_pull2,1)
            #
            # make sure all task code registered event, aka pulls, are within the video recording
            ind_good_pull1 = time_point_pull1 < (totalsess_time - session_start_time)
            time_point_pull1 = time_point_pull1[ind_good_pull1]
            ind_good_pull2 = time_point_pull2 < (totalsess_time - session_start_time)
            time_point_pull2 = time_point_pull2[ind_good_pull2]
            #
            ind_good_pull1_succ = time_point_pull1_succ < (totalsess_time - session_start_time)
            time_point_pull1_succ = time_point_pull1_succ[ind_good_pull1_succ]
            ind_good_pull2_succ = time_point_pull2_succ < (totalsess_time - session_start_time)
            time_point_pull2_succ = time_point_pull2_succ[ind_good_pull2_succ]
            #
            ind_good_pull1_fail = time_point_pull1_fail < (totalsess_time - session_start_time)
            time_point_pull1_fail = time_point_pull1_fail[ind_good_pull1_fail]
            ind_good_pull2_fail = time_point_pull2_fail < (totalsess_time - session_start_time)
            time_point_pull2_fail = time_point_pull2_fail[ind_good_pull2_fail]

            # 
            time_point_pulls_succfail = { "pull1_succ":time_point_pull1_succ,
                                          "pull2_succ":time_point_pull2_succ,
                                          "pull1_fail":time_point_pull1_fail,
                                          "pull2_fail":time_point_pull2_fail,
                                        }
            
            # 
            # based on time point pull and juice, define some features for each pull action
            pull_infos = get_pull_infos(animal1, animal2, time_point_pull1, time_point_pull2, 
                                        time_point_juice1, time_point_juice2, force_pull1, force_pull2,
                                        partnerforce_pull1, partnerforce_pull2)
            
            pull_infos_all_dates[date_tgt] = pull_infos
            
            # define variables and use them to find the 'onset' of pull decision
            print('project the high dimension bhv variables to PC space to define the start of the pull decision')
            #
            gausKernelsize = 16 # 4 or 16
            # clean the data
            time_point_pull1_temp = np.array(time_point_pull1)+session_start_time
            time_point_pull1_temp = time_point_pull1_temp[time_point_pull1_temp<totalsess_time]
            time_point_pull2_temp = np.array(time_point_pull2)+session_start_time
            time_point_pull2_temp = time_point_pull2_temp[time_point_pull2_temp<totalsess_time]
            #
            # organize the data into a time series
            pull1_data = np.zeros([int(totalsess_time*fps),])
            pull1_data[np.round(time_point_pull1_temp*fps).astype(int)]=1
            #
            pull2_data = np.zeros([int(totalsess_time*fps),])
            pull2_data[np.round(time_point_pull2_temp*fps).astype(int)]=1


            # get the continuous variables
            data_summary_twoanimals, data_summary_names = get_data_for_singlecam_conBhv_from_highDimension_to_PCspace(gausKernelsize, fps, animal1, animal2, 
                                                            animalnames_videotrack, session_start_time, 
                                                            time_point_pull1, time_point_pull2,
                                                            time_point_juice1, time_point_juice2, 
                                                            oneway_gaze1, oneway_gaze2, mutual_gaze1, mutual_gaze2, 
                                                            output_look_ornot, output_allvectors, 
                                                            output_allangles, output_key_locations)
            #
            vars_toPCA_names = ['gaze_other_angle', 'gaze_tube_angle', 'gaze_lever_angle', 'animal_animal_dist',
                                'animal_tube_dist', 'animal_lever_dist', 'mass_move_speed', 'gaze_angle_speed',]
            #
            indices = [data_summary_names.index(name) for name in vars_toPCA_names]
            # 
            # reduce to smaller dimension using PCA; for animal 1
            allbhvs_a1 = np.array(data_summary_twoanimals[animal1])[indices,:]
            data_for_pca = allbhvs_a1.T
            # Normalize the data (Z-score scaling)
            # Each of the 8 variables will now have a mean of 0 and a standard deviation of 1
            scaler = StandardScaler()
            data_scaled = scaler.fit_transform(data_for_pca)
            # Initialize and run PCA
            # We want to reduce the 8 variables to 3 principal components
            pca = PCA(n_components=3)
            # Fit the model and transform the data
            principal_components = pca.fit_transform(data_scaled)
            principal_components_transposed = principal_components.T
            # Get the explained variance for each component
            explained_variance_a1 = pca.explained_variance_ratio_
            #
            PC1_a1 = principal_components_transposed[0,:]
            # loadings
            PC1_a1_loadings = pca.components_[0]
            # 
            # reduce to smaller dimension using PCA; for animal 2
            allbhvs_a2 = np.array(data_summary_twoanimals[animal2])[indices,:]
            data_for_pca = allbhvs_a2.T
            # Normalize the data (Z-score scaling)
            # Each of the 8 variables will now have a mean of 0 and a standard deviation of 1
            scaler = StandardScaler()
            data_scaled = scaler.fit_transform(data_for_pca)
            # Initialize and run PCA
            # We want to reduce the 8 variables to 3 principal components
            pca = PCA(n_components=3)
            # Fit the model and transform the data
            principal_components = pca.fit_transform(data_scaled)
            principal_components_transposed = principal_components.T
            # Get the explained variance for each component
            explained_variance_a2 = pca.explained_variance_ratio_
            #
            PC1_a2 = principal_components_transposed[0,:]
            # loadings
            PC1_a2_loadings = pca.components_[0]
            #
            # put the PC1 loading and PC123 explained variance together and save 
            bhvtoPC1_loadings_all_dates[date_tgt][animal1] = PC1_a1_loadings
            bhvtoPC1_loadings_all_dates[date_tgt][animal2] = PC1_a2_loadings
            #
            bhvPC123explained_all_dates[date_tgt][animal1] = explained_variance_a1
            bhvPC123explained_all_dates[date_tgt][animal2] = explained_variance_a2

            #
            # use one of the two methods; not the HMM based method
            if not doHMMmethod:
                if not doOnsetAfterMin:
                    # find the transitional time point of angle speed and speed in IPI
                    speed1_increase = find_sharp_increases_withinIPI(pull1_data,PC1_a1,session_start_time,fps)
                    pull1_action_onset_frames = speed1_increase
                    #
                    speed2_increase = find_sharp_increases_withinIPI(pull2_data,PC1_a2,session_start_time,fps)
                    pull2_action_onset_frames = speed2_increase
                #
                elif doOnsetAfterMin:
                    # find the transitional time point of angle speed and speed in IPI
                    speed1_increase = find_rising_onset_after_min_withinIPI(pull1_data,PC1_a1,session_start_time,fps)
                    pull1_action_onset_frames = speed1_increase
                    #
                    speed2_increase = find_rising_onset_after_min_withinIPI(pull2_data,PC1_a2,session_start_time,fps)
                    pull2_action_onset_frames = speed2_increase
            #
            elif doHMMmethod:
                n_states = 3

                pull1_action_onset_framepoints, _ = get_trial_start_frames_from_HMM(speed1_data, anglespeed1_data, pull1_data, 
                                                                                    fps, session_start_time, n_states)
                pull1_action_onset_frames = np.isin(np.arange(len(pull1_data)), pull1_action_onset_framepoints).astype(int)
                #
                pull2_action_onset_framepoints, _ = get_trial_start_frames_from_HMM(speed2_data, anglespeed2_data, pull2_data, 
                                                                                    fps, session_start_time, n_states)
                pull2_action_onset_frames = np.isin(np.arange(len(pull2_data)), pull2_action_onset_framepoints).astype(int)


            #
            # store the pull reaction time information
            # temporary fix
            try:
                pull_data_points = np.where(pull1_data)[0]
                pullonset_data_points = np.where(pull1_action_onset_frames)[0]
                pull1_rt = (pull_data_points - pullonset_data_points)/fps
                pull_rts_all_dates[date_tgt][animal1] = pull1_rt
                #
                pull_data_points = np.where(pull2_data)[0]
                pullonset_data_points = np.where(pull2_action_onset_frames)[0]
                pull2_rt = (pull_data_points - pullonset_data_points)/fps
                pull_rts_all_dates[date_tgt][animal2] = pull2_rt
            except:
                continue

            #
            # replace time_point_pull_xxx to the pull onset
            time_point_pull1 = np.array(np.round(time_point_pull1,1))
            time_point_pull2 = np.array(np.round(time_point_pull2,1))
            time_point_pull1_succ = np.array(time_point_pull1_succ)
            time_point_pull2_succ = np.array(time_point_pull2_succ)
            time_point_pull1_fail = np.array(time_point_pull1_fail)
            time_point_pull2_fail = np.array(time_point_pull2_fail)
            #
            time_point_pull1_succ_idx = np.isin(time_point_pull1,time_point_pull1_succ)
            time_point_pull2_succ_idx = np.isin(time_point_pull2,time_point_pull2_succ)
            time_point_pull1_fail_idx = np.isin(time_point_pull1,time_point_pull1_fail)
            time_point_pull2_fail_idx = np.isin(time_point_pull2,time_point_pull2_fail)
            #
            time_point_pull1 = np.where(pull1_action_onset_frames)[0]/fps - session_start_time
            time_point_pull2 = np.where(pull2_action_onset_frames)[0]/fps - session_start_time
            #
            time_point_pull1_succ = time_point_pull1[time_point_pull1_succ_idx]
            time_point_pull2_succ = time_point_pull2[time_point_pull2_succ_idx]
            time_point_pull1_fail = time_point_pull1[time_point_pull1_fail_idx]
            time_point_pull2_fail = time_point_pull2[time_point_pull2_fail_idx]
            #
            pull1_rt_succ = pull1_rt[time_point_pull1_succ_idx]
            pull2_rt_succ = pull2_rt[time_point_pull2_succ_idx]
            pull1_rt_fail = pull1_rt[time_point_pull1_fail_idx]
            pull2_rt_fail = pull2_rt[time_point_pull2_fail_idx]
            
            # plot key continuous behavioral variables
            if 1:
                print('plot self pull start triggered bhv variables')

                filepath_cont_var = data_saved_folder+'bhv_events_continuous_variables_singlecam_wholebody/'+animal1_fixedorder[0]+animal2_fixedorder[0]+'/'+cameraID+'/'+date_tgt+'/'
                if not os.path.exists(filepath_cont_var):
                    os.makedirs(filepath_cont_var)

                min_length = np.shape(look_at_other_or_not_merge['dodson'])[0] # frame numbers of the video recording

                # NOTE! This one used the wrong and old version of separating successful and failed 
                pull_trig_events_summary = plot_continuous_bhv_var_singlecam_PullStartToPull_variedSection_highbhvDimension_to_lowPCspace(
                                        pull1_rt, pull2_rt, animal1, animal2, 
                                        session_start_time, min_length, succpulls_ornot, time_point_pull1, time_point_pull2, 
                                        oneway_gaze1, oneway_gaze2, mutual_gaze1, mutual_gaze2, animalnames_videotrack,
                                        output_look_ornot, output_allvectors, output_allangles,output_key_locations)
                pullstartTopull_trig_events_all_dates[date_tgt] = pull_trig_events_summary

                # successful pull
                try:
                    pull_trig_events_summary = plot_continuous_bhv_var_singlecam_PullStartToPull_variedSection_highbhvDimension_to_lowPCspace(
                                            pull1_rt_succ, pull2_rt_succ, animal1, animal2, 
                                            session_start_time, min_length, succpulls_ornot, 
                                            time_point_pull1_succ, time_point_pull2_succ, 
                                            oneway_gaze1, oneway_gaze2, mutual_gaze1, mutual_gaze2, animalnames_videotrack,
                                            output_look_ornot, output_allvectors, output_allangles,output_key_locations)
                    succpullstartTopull_trig_events_all_dates[date_tgt] = pull_trig_events_summary
                except:
                    succpullstartTopull_trig_events_all_dates[date_tgt] = np.nan

                # failed pull
                try:
                    pull_trig_events_summary = plot_continuous_bhv_var_singlecam_PullStartToPull_variedSection_highbhvDimension_to_lowPCspace(
                                            pull1_rt_fail, pull2_rt_fail, animal1, animal2, 
                                            session_start_time, min_length, succpulls_ornot, 
                                            time_point_pull1_fail, time_point_pull2_fail, 
                                            oneway_gaze1, oneway_gaze2, mutual_gaze1, mutual_gaze2, animalnames_videotrack,
                                            output_look_ornot, output_allvectors, output_allangles,output_key_locations)
                    failpullstartTopull_trig_events_all_dates[date_tgt] = pull_trig_events_summary
                except:
                    failpullstartTopull_trig_events_all_dates[date_tgt] = np.nan

                
        # after all the analysis, separate them based on different subblock    
        # get task type and cooperation threshold
        # tasktype: 1-normal SR, 2-force changed SR, 3-normal coop, 4-force changed coop
        trialID_list = np.array(trial_record_clean['trial_number'],dtype = 'int')
        tasktype_list = np.array(trial_record_clean['task_type'],dtype = 'int')
        coop_thres_list = np.array(trial_record_clean['pulltime_thres'],dtype = 'int')
        lever1force_list = np.array(trial_record_clean['lever1_force'],dtype = 'int')
        lever2force_list = np.array(trial_record_clean['lever2_force'],dtype = 'int')
        
        # use the combination of lever 1/2 forces to separate trials
        force12_uniques,indices = np.unique(np.vstack((lever1force_list,lever2force_list)),axis=1,return_index=True)
        force12_uniques = force12_uniques[:,np.argsort(indices)]
        ntrialtypes = np.shape(force12_uniques)[1]
        #
        # Convert columns to the desired string format
        force12_list = [f"{force12_uniques[0][i]}&{force12_uniques[1][i]}" for i in range(force12_uniques.shape[1])]
        
                
        # 
        for itrialtype in np.arange(0,ntrialtypes,1):
            force1_unique = force12_uniques[0,itrialtype]
            force2_unique = force12_uniques[1,itrialtype]

            ind = np.isin(lever1force_list,force1_unique) & np.isin(lever2force_list,force2_unique)
            
            trialID_itrialtype = trialID_list[ind]
            
            tasktype_itrialtype = np.unique(tasktype_list[ind])
            coop_thres_itrialtype = np.unique(coop_thres_list[ind])
            
            # save some simple measures
            animal1_name_all_dates = np.append(animal1_name_all_dates,animal1)
            animal2_name_all_dates = np.append(animal2_name_all_dates,animal2)
            trialdates_all_dates = np.append(trialdates_all_dates,date_tgt)
            tasktypes_all_dates = np.append(tasktypes_all_dates,tasktype_itrialtype)
            coopthres_all_dates = np.append(coopthres_all_dates,coop_thres_itrialtype)
            #
            if np.isin(animal1,animal1_fixedorder):
                force1_all_dates = np.append(force1_all_dates,force1_unique)
                force2_all_dates = np.append(force2_all_dates,force2_unique)
            else:
                force1_all_dates = np.append(force1_all_dates,force2_unique)
                force2_all_dates = np.append(force2_all_dates,force1_unique)
            #
            trialnum_all_dates = np.append(trialnum_all_dates,np.sum(ind))
            subblockID_all_dates = np.append(subblockID_all_dates,itrialtype)
            
            # analyze behavior results
            bhv_data_itrialtype = bhv_data[np.isin(bhv_data['trial_number'],trialID_itrialtype)]
            #
            # successful rates
            succ_rate_itrialtype = np.sum((bhv_data_itrialtype['behavior_events']==3)|(bhv_data_itrialtype['behavior_events']==4))/np.sum((bhv_data_itrialtype['behavior_events']==1)|(bhv_data_itrialtype['behavior_events']==2))
            succ_rate_all_dates = np.append(succ_rate_all_dates,succ_rate_itrialtype)
            #
            succ_rate1_itrialtype = np.sum((bhv_data_itrialtype['behavior_events']==3))/np.sum((bhv_data_itrialtype['behavior_events']==1))
            succ_rate2_itrialtype = np.sum((bhv_data_itrialtype['behavior_events']==4))/np.sum((bhv_data_itrialtype['behavior_events']==2))            
            if np.isin(animal1,animal1_fixedorder):
                succ_rate1_all_dates = np.append(succ_rate1_all_dates,succ_rate1_itrialtype)
                succ_rate2_all_dates = np.append(succ_rate2_all_dates,succ_rate2_itrialtype)
            else:
                succ_rate1_all_dates = np.append(succ_rate1_all_dates,succ_rate2_itrialtype)
                succ_rate2_all_dates = np.append(succ_rate2_all_dates,succ_rate1_itrialtype)
            
            # block time
            block_starttime = bhv_data_itrialtype[bhv_data_itrialtype['behavior_events']==0]['time_points'].iloc[0]
            block_endtime = bhv_data_itrialtype[bhv_data_itrialtype['behavior_events']==9]['time_points'].iloc[-1]
            blocktime_all_dates = np.append(blocktime_all_dates,block_endtime-block_starttime)
           
            #
            # across animal interpull interval
            pullid = np.array(bhv_data_itrialtype[(bhv_data_itrialtype['behavior_events']==1) | (bhv_data_itrialtype['behavior_events']==2)]["behavior_events"])
            pulltime = np.array(bhv_data_itrialtype[(bhv_data_itrialtype['behavior_events']==1) | (bhv_data_itrialtype['behavior_events']==2)]["time_points"])
            pullid_diff = np.abs(pullid[1:] - pullid[0:-1])
            pulltime_diff = pulltime[1:] - pulltime[0:-1]
            interpull_intv = pulltime_diff[pullid_diff==1]
            interpull_intv = interpull_intv[interpull_intv<20]
            mean_interpull_intv = np.nanmean(interpull_intv)
            std_interpull_intv = np.nanstd(interpull_intv)
            #
            interpullintv_all_dates = np.append(interpullintv_all_dates,mean_interpull_intv)
            # 
            # animal 1 and 2's pull numbers
            if np.isin(animal1,animal1_fixedorder):
                pull1_num_all_dates = np.append(pull1_num_all_dates,np.sum(bhv_data_itrialtype['behavior_events']==1))
                pull2_num_all_dates = np.append(pull2_num_all_dates,np.sum(bhv_data_itrialtype['behavior_events']==2))
            else:
                pull1_num_all_dates = np.append(pull1_num_all_dates,np.sum(bhv_data_itrialtype['behavior_events']==2))
                pull2_num_all_dates = np.append(pull2_num_all_dates,np.sum(bhv_data_itrialtype['behavior_events']==1))
            #
            # animal 1 and 2's within animal interpull interval
            pull1time = np.array(bhv_data_itrialtype[(bhv_data_itrialtype['behavior_events']==1)]["time_points"])
            ipi_pull1 = pull1time[1:]-pull1time[0:-1]
            ipi_pull1 = ipi_pull1[ipi_pull1<20]
            mean_ipi_pull1 = np.nanmean(ipi_pull1)
            std_ipi_pull1 = np.nanstd(ipi_pull1)/np.sqrt(np.shape(ipi_pull1)[0])
            pull2time = np.array(bhv_data_itrialtype[(bhv_data_itrialtype['behavior_events']==2)]["time_points"])
            ipi_pull2 = pull2time[1:]-pull2time[0:-1]
            ipi_pull2 = ipi_pull2[ipi_pull2<20]
            mean_ipi_pull2 = np.nanmean(ipi_pull2)
            std_ipi_pull2 = np.nanstd(ipi_pull2)/np.sqrt(np.shape(ipi_pull2)[0])
            if np.isin(animal1,animal1_fixedorder):
                pull1_IPI_all_dates = np.append(pull1_IPI_all_dates,mean_ipi_pull1)
                pull2_IPI_all_dates = np.append(pull2_IPI_all_dates,mean_ipi_pull2)
                pull1_IPI_std_all_dates = np.append(pull1_IPI_std_all_dates,std_ipi_pull1)
                pull2_IPI_std_all_dates = np.append(pull2_IPI_std_all_dates,std_ipi_pull2)
            else:
                pull1_IPI_all_dates = np.append(pull1_IPI_all_dates,mean_ipi_pull2)
                pull2_IPI_all_dates = np.append(pull2_IPI_all_dates,mean_ipi_pull1)
                pull1_IPI_std_all_dates = np.append(pull1_IPI_std_all_dates,std_ipi_pull2)
                pull2_IPI_std_all_dates = np.append(pull2_IPI_std_all_dates,std_ipi_pull1)
            
    
            # gaze number, based on the DLC tracking 
            if np.isin(animal1,animal1_fixedorder):
                owgaze1_num_all_dates = np.append(owgaze1_num_all_dates,np.shape(oneway_gaze1[(oneway_gaze1<=block_endtime)&(oneway_gaze1>=block_starttime)])[0])
                owgaze2_num_all_dates = np.append(owgaze2_num_all_dates,np.shape(oneway_gaze2[(oneway_gaze2<=block_endtime)&(oneway_gaze2>=block_starttime)])[0])
                mtgaze1_num_all_dates = np.append(mtgaze1_num_all_dates,np.shape(mutual_gaze1[(mutual_gaze1<=block_endtime)&(mutual_gaze1>=block_starttime)])[0])
                mtgaze2_num_all_dates = np.append(mtgaze2_num_all_dates,np.shape(mutual_gaze2[(mutual_gaze2<=block_endtime)&(mutual_gaze2>=block_starttime)])[0])
            else:
                owgaze1_num_all_dates = np.append(owgaze1_num_all_dates,np.shape(oneway_gaze2[(oneway_gaze2<=block_endtime)&(oneway_gaze2>=block_starttime)])[0])
                owgaze2_num_all_dates = np.append(owgaze2_num_all_dates,np.shape(oneway_gaze1[(oneway_gaze1<=block_endtime)&(oneway_gaze1>=block_starttime)])[0])
                mtgaze1_num_all_dates = np.append(mtgaze1_num_all_dates,np.shape(mutual_gaze2[(mutual_gaze2<=block_endtime)&(mutual_gaze2>=block_starttime)])[0])
                mtgaze2_num_all_dates = np.append(mtgaze2_num_all_dates,np.shape(mutual_gaze1[(mutual_gaze1<=block_endtime)&(mutual_gaze1>=block_starttime)])[0])
            
            
            
                
    # save data
    if 0:        
        data_saved_subfolder = data_saved_folder+'data_saved_singlecam_wholebody'+savefile_sufix+'/'+cameraID+'/'+animal1_fixedorder[0]+animal2_fixedorder[0]+'/'
        if not os.path.exists(data_saved_subfolder):
            os.makedirs(data_saved_subfolder)
                
        with open(data_saved_subfolder+'/animal1_name_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(animal1_name_all_dates, f)
        with open(data_saved_subfolder+'/animal2_name_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(animal2_name_all_dates, f)
        with open(data_saved_subfolder+'/trialdates_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(trialdates_all_dates, f)
        with open(data_saved_subfolder+'/tasktypes_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(tasktypes_all_dates, f)
        with open(data_saved_subfolder+'/coopthres_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(coopthres_all_dates, f)
            
        with open(data_saved_subfolder+'/force1_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(force1_all_dates, f)
        with open(data_saved_subfolder+'/force2_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(force2_all_dates, f)
            
        with open(data_saved_subfolder+'/subblockID_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(subblockID_all_dates, f)
            
        with open(data_saved_subfolder+'/succ_rate_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(succ_rate_all_dates, f)
        with open(data_saved_subfolder+'/succ_rate1_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(succ_rate1_all_dates, f)
        with open(data_saved_subfolder+'/succ_rate2_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(succ_rate2_all_dates, f)
            
        with open(data_saved_subfolder+'/trialnum_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(trialnum_all_dates, f)
        with open(data_saved_subfolder+'/blocktime_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(blocktime_all_dates, f)
        
        with open(data_saved_subfolder+'/interpullintv_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(interpullintv_all_dates, f)
        with open(data_saved_subfolder+'/pull1_IPI_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(pull1_IPI_all_dates, f)
        with open(data_saved_subfolder+'/pull1_IPI_std_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(pull1_IPI_std_all_dates, f)
        with open(data_saved_subfolder+'/pull2_IPI_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(pull2_IPI_all_dates, f)
        with open(data_saved_subfolder+'/pull2_IPI_std_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(pull2_IPI_std_all_dates, f)
                
        with open(data_saved_subfolder+'/owgaze1_num_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(owgaze1_num_all_dates, f)
        with open(data_saved_subfolder+'/owgaze2_num_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(owgaze2_num_all_dates, f)
        with open(data_saved_subfolder+'/mtgaze1_num_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(mtgaze1_num_all_dates, f)
        with open(data_saved_subfolder+'/mtgaze2_num_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(mtgaze2_num_all_dates, f)       
        with open(data_saved_subfolder+'/pull1_num_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(pull1_num_all_dates, f)
        with open(data_saved_subfolder+'/pull2_num_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(pull2_num_all_dates, f)
            
        with open(data_saved_subfolder+'/pullstartTopull_trig_events_all_dates_'+doOnsetAfterMin_suffix+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(pullstartTopull_trig_events_all_dates, f) 
        with open(data_saved_subfolder+'/succpullstartTopull_trig_events_all_dates_'+doOnsetAfterMin_suffix+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(succpullstartTopull_trig_events_all_dates, f) 
        with open(data_saved_subfolder+'/failpullstartTopull_trig_events_all_dates_'+doOnsetAfterMin_suffix+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(failpullstartTopull_trig_events_all_dates, f) 

        with open(data_saved_subfolder+'/bhvevents_pullstartTopull_aligned_FR_allevents_all_dates_'+doOnsetAfterMin_suffix+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(bhvevents_pullstartTopull_aligned_FR_allevents_all_dates, f) 
        
        with open(data_saved_subfolder+'/bhvtoPC1_loadings_all_dates_'+doOnsetAfterMin_suffix+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(bhvtoPC1_loadings_all_dates, f) 
        with open(data_saved_subfolder+'/bhvPC123explained_all_dates_'+doOnsetAfterMin_suffix+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(bhvPC123explained_all_dates, f) 
            
        with open(data_saved_subfolder+'/pull_rts_all_dates_'+doOnsetAfterMin_suffix+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(pull_rts_all_dates, f)  
        with open(data_saved_subfolder+'/pull_infos_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(pull_infos_all_dates, f)   


    # save a subset of the data
    if 1:   
        
        data_saved_subfolder = data_saved_folder+'data_saved_singlecam_wholebody'+savefile_sufix+'/'+cameraID+'/'+animal1_fixedorder[0]+animal2_fixedorder[0]+'/'
        if not os.path.exists(data_saved_subfolder):
            os.makedirs(data_saved_subfolder)
        
        with open(data_saved_subfolder+'/pullstartTopull_trig_events_all_dates_'+doOnsetAfterMin_suffix+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(pullstartTopull_trig_events_all_dates, f) 
        with open(data_saved_subfolder+'/succpullstartTopull_trig_events_all_dates_'+doOnsetAfterMin_suffix+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(succpullstartTopull_trig_events_all_dates, f) 
        with open(data_saved_subfolder+'/failpullstartTopull_trig_events_all_dates_'+doOnsetAfterMin_suffix+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(failpullstartTopull_trig_events_all_dates, f) 
        
        with open(data_saved_subfolder+'/bhvtoPC1_loadings_all_dates_'+doOnsetAfterMin_suffix+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(bhvtoPC1_loadings_all_dates, f) 
        with open(data_saved_subfolder+'/bhvPC123explained_all_dates_'+doOnsetAfterMin_suffix+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(bhvPC123explained_all_dates, f) 
                
        with open(data_saved_subfolder+'/pull_rts_all_dates_'+doOnsetAfterMin_suffix+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(pull_rts_all_dates, f)  
        with open(data_saved_subfolder+'/pull_infos_all_dates_'+animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+forceManiType+'.pkl', 'wb') as f:
            pickle.dump(pull_infos_all_dates, f) 
        
                        

### do some plotting
#### plot the correlation between rts and force level

In [ ]:
# step 1: prepare the data
rows = []

for date, animals_dict in pull_rts_all_dates.items():

    animals = list(animals_dict.keys())

    for self_animal in animals:

        partner_animal = [a for a in animals if a != self_animal][0]

        pull_rts = animals_dict[self_animal]
        n_pulls = len(pull_rts)

        info = pull_infos_all_dates[date]

        failpull = info[(self_animal, 'num_preceding_failpull')].reset_index(drop=True)
        t_last_rew = info[(self_animal, 'time_from_last_reward')].reset_index(drop=True)
        pull_interval = info[(self_animal, 'pull_interval')].reset_index(drop=True)
        lever_force = info[(self_animal, 'lever_force')].reset_index(drop=True)
        partner_lever_force = info[(self_animal, 'partner_lever_force')].reset_index(drop=True)
        
        #
        pullstartTopull_trig_events = pullstartTopull_trig_events_all_dates[date]

        self_gaze_trace = pullstartTopull_trig_events[(self_animal,'socialgaze_prob')]
        partner_PC1_trace = pullstartTopull_trig_events[(self_animal,'other_PC1')]

        for pull_id in range(n_pulls):

            
            # add social gaze and gaze-filtered social evidence information
            self_gaze_trace_ipull = self_gaze_trace[pull_id]
            partner_PC1_trace_ipull = partner_PC1_trace[pull_id]
            
            gaze_accum = float(np.sum(self_gaze_trace_ipull))
            
            partner_PC1_mean = float(np.nanmean(partner_PC1_trace_ipull[self_gaze_trace_ipull>0.05]))
            partner_PC1_std = float(np.nanstd(partner_PC1_trace_ipull[self_gaze_trace_ipull>0.05]))
            
            # pull things together
            rows.append({
                'date': date,
                'self_animal': self_animal,
                'partner_animal': partner_animal,
                'pull_id': pull_id,

                'pull_rt': pull_rts[pull_id],

                'self_force': lever_force.iloc[pull_id],
                'partner_force': partner_lever_force.iloc[pull_id],
                
                'num_preceding_failpull': failpull.iloc[pull_id],
                'time_from_last_reward': t_last_rew.iloc[pull_id],
                'pull_interval': pull_interval.iloc[pull_id],
                
                'gaze_accum': gaze_accum,
                'partner_PC1_mean': partner_PC1_mean,
                'partner_PC1_std': partner_PC1_std,
            })
            

big_pull_df = pd.DataFrame(rows)

# sort first so temporal order is correct within each animal
big_pull_df = big_pull_df.sort_values(['date', 'self_animal', 'pull_id'])

# create force pair
big_pull_df['force_pair'] = list(
    zip(big_pull_df['self_force'], big_pull_df['partner_force'])
)

# compute subblockID per (date × self_animal)
big_pull_df['subblockID'] = (
    big_pull_df
    .groupby(['date', 'self_animal'])['force_pair']
    .transform(lambda s: (s != s.shift()).cumsum() - 1)
)

# clean up
big_pull_df = big_pull_df.drop(columns='force_pair')

# make sure forces are numeric
big_pull_df['self_force'] = pd.to_numeric(big_pull_df['self_force'])
big_pull_df['partner_force'] = pd.to_numeric(big_pull_df['partner_force'])

# container columns
big_pull_df['delta_self_force_first'] = np.nan
big_pull_df['delta_self_force_prev'] = np.nan
big_pull_df['delta_partner_force_first'] = np.nan
big_pull_df['delta_partner_force_prev'] = np.nan

# compute deltas per (date × self_animal)
for (date, animal), df in big_pull_df.groupby(['date', 'self_animal']):

    df = df.sort_values('pull_id')

    # block-level force (one value per block)
    block_force = (
        df.groupby('subblockID')[['self_force','partner_force']]
        .first()
        .sort_index()
    )

    # Δ to first block
    delta_first = block_force - block_force.iloc[0]

    # Δ to previous block
    delta_prev = block_force.diff().fillna(0)

    # map back to pulls
    big_pull_df.loc[df.index, 'delta_self_force_first'] = \
        df['subblockID'].map(delta_first['self_force'])

    big_pull_df.loc[df.index, 'delta_self_force_prev'] = \
        df['subblockID'].map(delta_prev['self_force'])

    big_pull_df.loc[df.index, 'delta_partner_force_first'] = \
        df['subblockID'].map(delta_first['partner_force'])

    big_pull_df.loc[df.index, 'delta_partner_force_prev'] = \
        df['subblockID'].map(delta_prev['partner_force'])

In [ ]:
# ===============================
# STEP 2 — PLOTTING
# ===============================

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import scipy.stats as st

plot_df = big_pull_df.copy()

# -------------------------------
# Remove pull_rt outliers (per date × animal)
# -------------------------------
def iqr_filter(group):

    q1 = group['pull_rt'].quantile(0.25)
    q3 = group['pull_rt'].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return group[(group['pull_rt'] >= lower) &
                 (group['pull_rt'] <= upper)]

plot_df = (
    plot_df
    .groupby(['date', 'self_animal'], group_keys=False)
    .apply(iqr_filter)
    .reset_index(drop=True)
)

plot_df = plot_df.replace([np.inf, -np.inf], np.nan)

# -------------------------------
# Focus on ONE animal
# -------------------------------
animal_plot_tgt = animal1_fixedorder[0]

plot_df = plot_df[plot_df['self_animal'] == animal_plot_tgt]

# -------------------------------
# Residualize pull_rt over subblockID (within date)
# -------------------------------
plot_df['pull_rt_resid'] = np.nan

for date, df in plot_df.groupby('date'):

    df = df.sort_values('pull_id')

    if df['subblockID'].nunique() > 1:

        slope, intercept, *_ = st.linregress(
            df['subblockID'], df['pull_rt']
        )

        resid = df['pull_rt'] - (slope * df['subblockID'] + intercept)

        plot_df.loc[df.index, 'pull_rt_resid'] = resid

# -------------------------------
# VARIABLES
# -------------------------------
x_vars = [
    'self_force',
    'partner_force',
    'time_from_last_reward',
    'pull_interval',
    'subblockID',
    'delta_self_force_first',
    'delta_self_force_prev',
    'delta_partner_force_first',
    'delta_partner_force_prev',
    'gaze_accum',
    'partner_PC1_mean',
    'partner_PC1_std',
]

y_vars_extra = [
    'gaze_accum',
    'partner_PC1_mean',
    'partner_PC1_std'
]

# -------------------------------
# FIGURE SETUP
# -------------------------------
n_rows = 2 + len(y_vars_extra)
n_cols = len(x_vars)

fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(5*n_cols, 4*n_rows),
                         sharey=False)

# -------------------------------
# PLOTTING LOOP
# -------------------------------
for j, x in enumerate(x_vars):

    # ==========================================================
    # ROW 1 — RAW pull_rt
    # ==========================================================
    ax = axes[0, j]

    data = plot_df[[x, 'pull_rt', 'date']].dropna()

    if len(data) > 5:

        sns.regplot(data=data, x=x, y='pull_rt', ax=ax,
                    scatter_kws={'s': 15, 'alpha': 0.6},
                    line_kws={'color': 'red'})

        data_lmm = data.rename(columns={x: 'x', 'pull_rt': 'y'})
        data_lmm['date'] = data_lmm['date'].astype('category')

        try:
            fit = smf.mixedlm("y ~ x", data_lmm,
                              groups=data_lmm["date"]).fit(disp=False)

            beta = fit.params['x']
            pval = fit.pvalues['x']

        except:
            beta, pval = np.nan, np.nan

        ax.text(0.05, 0.95,
                f"β={beta:.2f}\np={pval:.3f}",
                transform=ax.transAxes, va='top')

    ax.set_title(f"raw pull_rt: {x}")

    # ==========================================================
    # ROW 2 — BLOCK-REGRESSED pull_rt
    # ==========================================================
    ax = axes[1, j]

    data = plot_df[[x, 'pull_rt_resid', 'date']].dropna()

    if len(data) > 5:

        sns.regplot(data=data, x=x, y='pull_rt_resid', ax=ax,
                    scatter_kws={'s': 15, 'alpha': 0.6},
                    line_kws={'color': 'red'})

        data_lmm = data.rename(columns={x: 'x', 'pull_rt_resid': 'y'})
        data_lmm['date'] = data_lmm['date'].astype('category')

        try:
            fit = smf.mixedlm("y ~ x", data_lmm,
                              groups=data_lmm["date"]).fit(disp=False)

            beta = fit.params['x']
            pval = fit.pvalues['x']

        except:
            beta, pval = np.nan, np.nan

        ax.text(0.05, 0.95,
                f"β={beta:.2f}\np={pval:.3f}",
                transform=ax.transAxes, va='top')

    ax.set_title(f"block-regressed pull_rt: {x}")

    # ==========================================================
    # ROWS 3–5 — SOCIAL / PC1 METRICS
    # ==========================================================
    for i_extra, y_extra in enumerate(y_vars_extra):

        ax = axes[2 + i_extra, j]

        data = plot_df[[x, y_extra, 'date']].dropna()
        
        if x == y_extra:
            ax.axis('off')
            continue

        if len(data) > 5:

            sns.regplot(data=data, x=x, y=y_extra, ax=ax,
                        scatter_kws={'s': 15, 'alpha': 0.6},
                        line_kws={'color': 'red'})

            data_lmm = data.rename(columns={x: 'x', y_extra: 'y'})
            data_lmm['date'] = data_lmm['date'].astype('category')

            try:
                fit = smf.mixedlm("y ~ x", data_lmm,
                                  groups=data_lmm["date"]).fit(disp=False)

                beta = fit.params['x']
                pval = fit.pvalues['x']

            except:
                beta, pval = np.nan, np.nan

            ax.text(0.05, 0.95,
                    f"β={beta:.2f}\np={pval:.3f}",
                    transform=ax.transAxes, va='top')

        ax.set_title(f"{y_extra}: {x}")

# -------------------------------
# FINAL TOUCH
# -------------------------------
plt.suptitle(f"{animal_plot_tgt} — {forceManiType}", y=1.02, fontsize=16)
plt.tight_layout()
# plt.show()